In [5]:
#Thomas Winn
#Wisconsin Space Program (WiSP)
#Fanno Flow Model

import math
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, solve, log
from scipy.optimize import fsolve

solver_option = int(input("Input a number to select solver option. 1 = Maximum (choked) mass flow for set conditions, 2 = Input inlet mach number, 3 = Input inlet mass flow rate"))

print()

print("Note: 'Feature 1' is the length of pipe or the equivilant length of the joint between the inlet and the end of the first pipe.")
print("All other 'Feature' designations follow this rule")

print()

pipe_features = int(input("How many pipe features are in the system? (#Pipe lenghts + Pipe features (like bends))"))

print()

inlet_pressure = 4826337
specific_heat_ratio = 1.4
temperature = 200 #kelvin
roughness = 2.5 * (10**-6) #meters
diameter = 0.009525 #meters
dynamic_visc = 15 * (10**-6)
mach_speed = math.sqrt(specific_heat_ratio * 296.77 * temperature) #m/s
reynolds_num = (mach_speed * diameter) / dynamic_visc

def darcey_function(f):
    return (-2*(np.log10(((roughness/diameter)/3.7) + (2.51 / (reynolds_num * (np.sqrt(f)))))) - (1/np.sqrt(f)))

darcey_friction_factor = fsolve(darcey_function, 0.01)[0] 

print(darcey_friction_factor)

def fanno_function(M):
    return ((((1 - (M**2)) / (1.4 * (M**2)))) + ((2.4/2.8) * np.log((2.4 * (M**2)) / (2 + (0.4 * (M**2))))))

def mach_calc(fanno_length):
    func = lambda M: fanno_function(M) - fanno_length

    solutions = fsolve(func, 0.5)
    return abs(solutions[0])


print()

def pressure_calc(pressure1, mach1, mach2):
    pressure2 = pressure1 * (((1/mach2) * (1/(math.sqrt((2 / (1.40 + 1)) * (1 + (((1.40-1)/2) * (mach2**2))))))) / ((1/mach1) * (1/(math.sqrt((2 / (1.40 + 1)) * (1 + (((1.40-1)/2) * (mach1**2))))))))
    return pressure2

def density_calc(density1, mach1, mach2):
    density2 = density1 * (((1/mach2) * math.sqrt((2 / (1.40 + 1)) * (1 + (((1.40-1)/2) * (mach2**2))))) / ((1/mach1) * math.sqrt((2 / (1.40 + 1)) * (1 + (((1.40-1)/2) * (mach1**2))))))
    return density2

def temp_calc(temp1, mach1, mach2):
    temp2 = temp1 * ((1/((2 / (1.40 + 1)) * (1 + (((1.40-1)/2) * (mach2**2))))) / (1/((2 / (1.40 + 1)) * (1 + (((1.40-1)/2) * (mach1**2))))))
    return temp2

def velocity_calc(velocity1, mach1, mach2):
    velocity2 = velocity1* ((mach2 * (1/(math.sqrt((2/(1.40+1)) * (1 + (((1.40-1)/2) * (mach2**2))))))) / (mach1 * (1/(math.sqrt((2/(1.40+1)) * (1 + (((1.40-1)/2) * (mach1**2))))))))
    return velocity2

def stag_pressure_calc(stag_pressure1, mach1, mach2):
    stag_pressure2 = stag_pressure1 * (((1/mach2) * (((2/(1.40+1)) * (1 + (((1.40-1)/2) * (mach2**2))))**((1.40+1) / (2*(1.40-1))))) / ((1/mach1) * (((2/(1.40+1)) * (1 + (((1.40-1)/2) * (mach1**2))))**((1.40+1) / (2*(1.40-1))))))
    return stag_pressure2



def printout():
    
    print("Static Pressures:")
    for idx in range (len(static_pressures)):
        print("Static pressure at feature " + str(idx) + ": " + str(static_pressures[idx]) + " Pa")
    
    print()
    
    print("Dynamic Pressures:")
    for idx in range (len(dynamic_pressures)):
        print("Dynamic pressure at features " + str(idx) + ": " + str(dynamic_pressures[idx]) + " Pa")
    
    print()
    
    print("Stagnation Pressures:")
    for idx in range (len(stagnation_pressures)):
        print("Stagnation pressure at feature " + str(idx) + ": " + str(stagnation_pressures[idx]) + " Pa")
    
    print()
    
    print("Densities:")
    for idx in range (len(densities)):
        print("Density at feature " + str(idx) + ": " + str(densities[idx]) + " Kg/m^3")
    
    print()
    
    print("Temperatures:")
    for idx in range (len(temperatures)):
        print("Temperature at feature " + str(idx) + ": " + str(temperatures[idx]) + " K")
    
    print()
    
    print("Velocities:")
    for idx in range (len(velocities)):
        print("Velocity at feature " + str(idx) + ": " + str(velocities[idx]) + " m/s")
    
    print()

    print("Mach Numbers:")
    for idx in range (len(mach_numbers)):
        print("Mach number at feature " + str(idx) + ": " + str(mach_numbers[idx]))

    print()
    
    print("Mach Speeds:")
    for idx in range (len(local_mach_speeds)):
        print("Mach speed at feature " + str(idx) + ": " + str(local_mach_speeds[idx]) + " m/s")
    
    print()
    
    print("Mass flow rates: (should be equivilant)")
    for idx in range (len(mass_flow_rates)):
        print("Mass flow rate at feature " + str(idx) + ": " + str(mass_flow_rates[idx]) + " Kg/s")
    
    print()
    
    print("Percentage Pressure Loss: " + str(((inlet_pressure - stagnation_pressures[-1]) / (inlet_pressure)) * 100) + "%")


def calculate():

    global mass_flow_rates
    mass_flow_rates = []
    global local_mach_speeds
    local_mach_speeds = []
    global static_pressures
    static_pressures = []
    global densities
    densities = []
    global temperatures
    temperatures = []
    global velocities
    velocities = []
    global stagnation_pressures
    stagnation_pressures = []  
    global mach_numbers
    mach_numbers = []
    global dynamic_pressures
    dynamic_pressures = []
    
    if solver_option == 1:

        fanno_length = 0
        features = []
        
        for feature in range (pipe_features):
            length = float(input("What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length."))
            f_length = ((4 * darcey_friction_factor * length) / diameter)
            features.append(f_length)
            fanno_length += f_length

        print(fanno_length)
        
        print()
        
        inlet_mach = abs(mach_calc(fanno_length))
        
        fanno_length_remaining = fanno_length
        index = 1
        
        print("Mach Numbers")
        mach_numbers = []
        mach_numbers.append(inlet_mach)
        print("Mach number at inlet: " + str(inlet_mach))
        for feature in features:
        
            fanno_length_remaining -= feature
            print("Mach number at end of feature " + str(index) + ": " + str(abs(mach_calc(fanno_length_remaining))))
            mach_numbers.append(abs(mach_calc(fanno_length_remaining)))
            index +=1
        
        print()
        
        for idx in range (len(mach_numbers)):
            if idx == 0:
                static_pressures.append((inlet_pressure/(((1 + (((1.40-1)/2) * (mach_numbers[idx]**2)))**(1.40/(1.40-1))))))
                temperatures.append((temperature/((1 + (((1.40-1)/2) * (mach_numbers[idx]**2))))))
                densities.append((static_pressures[idx] * (28.02 / 1000)) / (8.314 * temperatures[idx]))
                local_mach_speeds.append(math.sqrt(1.40 * 296.8 * temperatures[idx]))
                velocities.append(mach_numbers[idx] * local_mach_speeds[idx])
                stagnation_pressures.append(inlet_pressure)
                mass_flow_rates.append(densities[idx] * velocities[idx] * (math.pi * ((diameter/2)**2)))
            else:
                static_pressures.append(pressure_calc(static_pressures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                stagnation_pressures.append(stag_pressure_calc(stagnation_pressures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                temperatures.append(temp_calc(temperatures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                velocities.append(velocity_calc(velocities[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                densities.append(density_calc(densities[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                mass_flow_rates.append(densities[idx] * velocities[idx] * (math.pi * ((diameter/2)**2)))
                local_mach_speeds.append(math.sqrt(1.40 * 296.8 * temperatures[idx]))
        
        dynamic_pressures = (np.array(stagnation_pressures) - np.array(static_pressures))

        printout()

    elif solver_option == 2:
        fanno_length = 0
        features = []
        
        for feature in range (pipe_features):
            length = float(input("What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length."))
            f_length = ((4 * darcey_friction_factor * length) / diameter)
            features.append(f_length)
            fanno_length += f_length
            print(fanno_length)
            print(fanno_function(selected_mach))
        
        if fanno_length > fanno_function(selected_mach):
            print("Error, flow chokes before fluid outlet. Please input a smaller mach number.")
        else:
            
            mach_numbers.append(selected_mach)

            fanno_length = fanno_function(selected_mach)
            remaining_fanno_lengths = [fanno_length]
            features_array = np.array(features)

            for idx in range (len(features)):
                remaining_fanno_lengths.append(fanno_length - (np.sum(features_array[0:(idx+1)])))

            for fanno in remaining_fanno_lengths[1:]:
                mach_numbers.append(mach_calc(fanno))
            
            for idx in range (len(features) + 1):
                
                if idx == 0:

                    static_pressures.append((inlet_pressure/(((1 + (((1.40-1)/2) * (mach_numbers[idx]**2)))**(1.40/(1.40-1))))))
                    temperatures.append((temperature/((1 + (((1.40-1)/2) * (mach_numbers[idx]**2))))))
                    densities.append((static_pressures[idx] * (28.02 / 1000)) / (8.314 * temperatures[idx]))
                    local_mach_speeds.append(math.sqrt(1.40 * 296.8 * temperatures[idx]))
                    velocities.append(mach_numbers[idx] * local_mach_speeds[idx])
                    stagnation_pressures.append(inlet_pressure)
                    mass_flow_rates.append(densities[idx] * velocities[idx] * (math.pi * ((diameter/2)**2)))
                    
                else:

                    static_pressures.append(pressure_calc(static_pressures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                    stagnation_pressures.append(stag_pressure_calc(stagnation_pressures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                    temperatures.append(temp_calc(temperatures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                    velocities.append(velocity_calc(velocities[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                    densities.append(density_calc(densities[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                    mass_flow_rates.append(densities[idx] * velocities[idx] * (math.pi * ((diameter/2)**2)))
                    local_mach_speeds.append(math.sqrt(1.40 * 296.8 * temperatures[idx]))

            printout()

    elif solver_option == 3:
        fanno_length = 0
        features = []
        
        for feature in range (pipe_features):
            length = float(input("What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length."))
            f_length = ((4 * darcey_friction_factor * length) / diameter)
            features.append(f_length)
            fanno_length += f_length

        print()
        
        def check_mass_flow(M):
            return (((inlet_pressure * (math.pi * ((diameter/2)**2))) / (math.sqrt(temperature))) * (math.sqrt(1.40/296.8)) * (M / ((1 + ((((1.40-1)/2) * (M**2))))**((1.40 + 1) / (2 * (1.40 - 1))))))
        solutions = (fsolve((lambda M: check_mass_flow(M) - selected_mass_flow), 0.05))

        mach_numbers.append(abs(solutions[0]))
        
        if fanno_length > fanno_function(mach_numbers[0]):
            print("Error: Selected mass flow rate exceeds maximium flow rate.")
            return
        
        fanno_length = fanno_function(mach_numbers[0])
        remaining_fanno_lengths = [fanno_length]
        features_array = np.array(features)

        for idx in range (len(features)):
            remaining_fanno_lengths.append(fanno_length - (np.sum(features_array[0:(idx+1)])))

        for fanno in remaining_fanno_lengths[1:]:
            mach_numbers.append(mach_calc(fanno))

        for idx in range (len(features) + 1):
            
            if idx == 0:

                static_pressures.append((inlet_pressure/(((1 + (((1.40-1)/2) * (mach_numbers[idx]**2)))**(1.40/(1.40-1))))))
                temperatures.append((temperature/((1 + (((1.40-1)/2) * (mach_numbers[idx]**2))))))
                densities.append((static_pressures[idx] * (28.02 / 1000)) / (8.314 * temperatures[idx]))
                local_mach_speeds.append(math.sqrt(1.40 * 296.8 * temperatures[idx]))
                velocities.append(mach_numbers[idx] * local_mach_speeds[idx])
                stagnation_pressures.append(inlet_pressure)
                mass_flow_rates.append(densities[idx] * velocities[idx] * (math.pi * ((diameter/2)**2)))
                
            else:

                static_pressures.append(pressure_calc(static_pressures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                stagnation_pressures.append(stag_pressure_calc(stagnation_pressures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                temperatures.append(temp_calc(temperatures[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                velocities.append(velocity_calc(velocities[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                densities.append(density_calc(densities[idx-1], mach_numbers[idx-1], mach_numbers[idx]))
                mass_flow_rates.append(densities[idx] * velocities[idx] * (math.pi * ((diameter/2)**2)))
                local_mach_speeds.append(math.sqrt(1.40 * 296.8 * temperatures[idx]))

        printout()
            

                    

if solver_option == 1:
    calculate()


# Inlet mass flow case:
elif solver_option == 2:
    
    selected_mach = float(input("What is the pipe's inlet mach number? (0-1)"))
    calculate()

elif solver_option == 3:

    selected_mass_flow = float(input("What is the pipe inlet's mass flow rate (kg/s)?"))
    calculate()




Input a number to select solver option. 1 = Maximum (choked) mass flow for set conditions, 2 = Input inlet mach number, 3 = Input inlet mass flow rate 3



Note: 'Feature 1' is the length of pipe or the equivilant length of the joint between the inlet and the end of the first pipe.
All other 'Feature' designations follow this rule



How many pipe features are in the system? (#Pipe lenghts + Pipe features (like bends)) 12



0.017689358321434864



What is the pipe inlet's mass flow rate (kg/s)? 0.056
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.1402842
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.9
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.046355
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 1.1
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.0889
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.9
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.8424418
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.9
What is the pipe's length (meters)? If it is a feature (like a bend), input an equivilant length. 0.069088
Wha


Static Pressures:
Static pressure at feature 0: 4822535.55014334 Pa
Static pressure at feature 1: 4818567.229013883 Pa
Static pressure at feature 2: 4793029.908547018 Pa
Static pressure at feature 3: 4791710.899103783 Pa
Static pressure at feature 4: 4760303.38981824 Pa
Static pressure at feature 5: 4757756.011774077 Pa
Static pressure at feature 6: 4731889.600209419 Pa
Static pressure at feature 7: 4707548.292273296 Pa
Static pressure at feature 8: 4681403.684081139 Pa
Static pressure at feature 9: 4679390.6533993855 Pa
Static pressure at feature 10: 4661871.675326357 Pa
Static pressure at feature 11: 4659689.614547451 Pa
Static pressure at feature 12: 4601967.628331093 Pa

Dynamic Pressures:

Stagnation Pressures:
Stagnation pressure at feature 0: 4826337 Pa
Stagnation pressure at feature 1: 4822371.809899773 Pa
Stagnation pressure at feature 2: 4796854.762590227 Pa
Stagnation pressure at feature 3: 4795536.80613081 Pa
Stagnation pressure at feature 4: 4764154.54230638 Pa
Stagnation